# Task 3.3 | CVM uplift

> **CVM:** What was the true effect of the August Plus push? Should it go to the entire base next week?

**The August push increased the 30-day Plus subscription rate by 6.11 percentage points**, from **6.57% in the holdout** to **12.68% in treatment**. That’s roughly double the holdout rate. Send the push again next week to eligible users, keep a randomized holdout, and test less-active users separately before expanding to the entire base.

| August test group | Assigned users | Users who subscribed | Plus subscription rate |
|---|---:|---:|---:|
| Treatment | 560 | 71 | 12.68% |
| Randomized holdout | 137 | 9 | 6.57% |

**Rollout decision.** Start with users who meet the August eligibility threshold of at least three successful transactions. At extract, **1,514 users** met that threshold, had no active Plus subscription, and hadn’t been assigned to the August test. Check eligibility and push reachability again before sending. If the audience size holds, an 80/20 split would assign **1,211 to treatment** and **303 to holdout**.

**Expected impact of the next push.** If users behave as they did in August, about **80 of the 1,211 users** would subscribe without a push, and about **154** would subscribe if we send it. That’s **74 additional Plus subscriptions** expected from the push, with the other **303 eligible users** in a randomized holdout. If we instead send it to all **1,514 eligible users**, we’d expect about **~92 additional subscriptions** under the same assumption.

**How we measured subscriptions.** For each group, divide the number of assigned users who subscribed from **August 16 through September 14** by the total number of users assigned to that group. Each user counts once. The campaign launched on August 15, and the subscription data records dates only. We used the next 30 calendar days for both groups because we couldn’t tell which August 15 subscriptions began after the push.

## Follow-up analysis and experiments

**Trace what users did after the push.** We treat the August push as a push notification. The existing data can show the time from send to the user’s next transaction. To understand the path to subscription, request notification delivery and open events, Plus screen views, and precise subscription timestamps. The holdout estimates how many subscriptions the push added overall. The event logs show how users engaged with the offer.

**Test an in-app banner and measure user response.** Randomly assign eligible users to a push notification, a banner shown when they open the app, or a no-message holdout. Compare Plus screen views and subscription rates across assigned groups. Among users who started with notifications enabled, compare how often they switched them off to see whether the push increased opt-outs.

**Measure cost per incremental paid subscription.** Divide the additional push delivery and infrastructure egress costs by the estimated number of paid subscriptions added versus the holdout. If the projected 74 incremental subscriptions each generate one paid month at EGP 49, that would be about **EGP 3,626 in gross subscription revenue**. Use actual payments, renewals, refunds, and campaign costs to assess whether the rollout pays back.

In [1]:
# Task 3.3 | August Lucky Plus campaign
from pathlib import Path

import pandas as pd
from sqlalchemy import create_engine

ROOT = next(
    folder for folder in (Path.cwd(), Path.cwd().parent)
    if (folder / "data" / "cleaned").is_dir()
)
CLEANED = ROOT / "data" / "cleaned"
CAMPAIGN_ID = "plus_push_aug"
LAUNCH_DATE = "2026-08-15"
PRIMARY_WINDOW_DAYS = 30
AS_OF_DATE = "2026-11-05"
MIN_SUCCESSFUL_TRANSACTIONS = 3
TREATMENT_SHARE = 0.80

campaigns = pd.read_csv(CLEANED / "cvm_campaigns.csv")
subscriptions = pd.read_csv(CLEANED / "plus_subscriptions.csv")
users = pd.read_csv(CLEANED / "users.csv", usecols=["user_id", "signup_ts"])
transactions = pd.read_csv(
    CLEANED / "transactions.csv", usecols=["user_id", "txn_ts", "status"]
)
settings = pd.DataFrame([{
    "campaign_id": CAMPAIGN_ID, "launch_date": LAUNCH_DATE,
    "followup_days": PRIMARY_WINDOW_DAYS,
    "as_of_date": AS_OF_DATE,
    "min_transactions": MIN_SUCCESSFUL_TRANSACTIONS,
    "treatment_share": TREATMENT_SHARE,
}])

engine = create_engine("sqlite://")
campaigns.to_sql("cvm_campaigns", engine, index=False)
subscriptions.to_sql("plus_subscriptions", engine, index=False)
users.to_sql("users", engine, index=False)
transactions.to_sql("transactions", engine, index=False)
settings.to_sql("settings", engine, index=False)

%load_ext sql
%sql engine

The 'toml' package isn't installed. To load settings from pyproject.toml or ~/.jupysql/config, install with: pip install toml

In [2]:
%%sql
-- Keep one outcome per assigned user and use the same follow-up in both groups.
DROP VIEW IF EXISTS temp.august_group_rates;

CREATE TEMP VIEW august_group_rates AS
WITH user_outcomes AS (
    SELECT c."group" AS campaign_group,
           EXISTS (
               SELECT 1
               FROM plus_subscriptions AS p
               WHERE p.user_id = c.user_id
                 AND p.plus_start_date > s.launch_date
                 AND p.plus_start_date <= DATE(s.launch_date, '+' || s.followup_days || ' days')
           ) AS started_plus
    FROM cvm_campaigns AS c JOIN settings AS s USING (campaign_id)
)
SELECT campaign_group, COUNT(*) AS assigned_users, SUM(started_plus) AS plus_starts,
       AVG(started_plus) AS subscription_rate
FROM user_outcomes
GROUP BY campaign_group;

SELECT r.campaign_group, r.assigned_users, r.plus_starts,
       ROUND(100 * r.subscription_rate, 2) AS subscription_rate_pct,
       ROUND(100 * (r.subscription_rate - h.subscription_rate), 2) AS lift_vs_holdout_pp
FROM august_group_rates AS r
JOIN august_group_rates AS h ON h.campaign_group = 'holdout'
ORDER BY r.campaign_group = 'holdout';

Running query in 'sqlite://'

campaign_group,assigned_users,plus_starts,subscription_rate_pct,lift_vs_holdout_pp
treatment,560,71,12.68,6.11
holdout,137,9,6.57,0.0


In [3]:
%%sql
-- Size the fresh test as of the extract. Active subscribers are excluded.
WITH eligible_users AS (
    SELECT t.user_id
    FROM transactions AS t CROSS JOIN settings AS s
    WHERE t.status = 'success' AND t.txn_ts < DATE(s.as_of_date, '+1 day')
    GROUP BY t.user_id
    HAVING COUNT(*) >= MAX(s.min_transactions)
),
audience AS (
    SELECT COALESCE(c."group", 'never assigned') AS august_assignment,
           COUNT(*) AS eligible_without_active_plus
    FROM users AS u
    JOIN eligible_users AS e USING (user_id)
    CROSS JOIN settings AS s
    LEFT JOIN cvm_campaigns AS c ON c.user_id = u.user_id AND c.campaign_id = s.campaign_id
    WHERE u.signup_ts < DATE(s.as_of_date, '+1 day')
      AND NOT EXISTS (
          SELECT 1
          FROM plus_subscriptions AS p
          WHERE p.user_id = u.user_id AND p.plus_start_date <= s.as_of_date
            AND (p.plus_end_date IS NULL OR p.plus_end_date > s.as_of_date)
      )
    GROUP BY august_assignment
),
allocation AS (
    SELECT a.*,
           CASE WHEN august_assignment = 'never assigned'
                THEN CAST(ROUND(eligible_without_active_plus * treatment_share) AS INTEGER)
                ELSE 0 END AS proposed_treatment,
           CASE WHEN august_assignment = 'never assigned'
                THEN eligible_without_active_plus ELSE 0 END AS test_size
    FROM audience AS a CROSS JOIN settings
)
SELECT august_assignment, eligible_without_active_plus, proposed_treatment,
       test_size - proposed_treatment AS proposed_holdout,
       CASE WHEN test_size > 0 THEN ROUND(proposed_treatment * t.subscription_rate) END
           AS projected_treatment_starts,
       CASE WHEN test_size > 0 THEN ROUND((test_size - proposed_treatment) * h.subscription_rate) END
           AS projected_holdout_starts,
       CASE WHEN test_size > 0 THEN ROUND(proposed_treatment * h.subscription_rate) END
           AS treatment_starts_without_push,
       CASE WHEN test_size > 0 THEN ROUND(proposed_treatment * (t.subscription_rate - h.subscription_rate)) END
           AS projected_incremental_starts
FROM allocation
JOIN august_group_rates AS t ON t.campaign_group = 'treatment'
JOIN august_group_rates AS h ON h.campaign_group = 'holdout'
ORDER BY CASE august_assignment WHEN 'never assigned' THEN 1 WHEN 'holdout' THEN 2 ELSE 3 END;

Running query in 'sqlite://'

august_assignment,eligible_without_active_plus,proposed_treatment,proposed_holdout,projected_treatment_starts,projected_holdout_starts,treatment_starts_without_push,projected_incremental_starts
never assigned,1514,1211,303,154.0,20.0,80.0,74.0
holdout,131,0,0,None,None,None,None
treatment,513,0,0,None,None,None,None


In [4]:
# Pearson chi-square test of campaign assignment vs. starting Plus within 30 days.
from math import erfc, sqrt
from IPython.display import display

observed = pd.read_sql_query(
    '''
    SELECT campaign_group, plus_starts AS subscribed,
           assigned_users - plus_starts AS did_not_subscribe
    FROM august_group_rates
    ''',
    engine,
    index_col="campaign_group",
).loc[["treatment", "holdout"]]

# Under independence, expected count = row total * column total / grand total.
expected = pd.DataFrame(
    observed.sum(axis=1).to_numpy()[:, None] * observed.sum(axis=0).to_numpy()
    / observed.to_numpy().sum(),
    index=observed.index,
    columns=observed.columns,
)
chi_square = (((observed - expected) ** 2) / expected).to_numpy().sum()

# A 2x2 table has one degree of freedom. This is its chi-square upper-tail probability.
p_value = erfc(sqrt(chi_square / 2))

display(observed)
print(f"Pearson chi-square (df=1, no continuity correction): {chi_square:.3f}")
print(f"p-value: {p_value:.4f} | Smallest expected count: {expected.to_numpy().min():.2f}")
print("The p-value is close to 0.05. Interpret it alongside the estimated lift.")

,subscribed,did_not_subscribe
campaign_group,,
treatment,71,489
holdout,9,128


Pearson chi-square (df=1, no continuity correction): 4.043
p-value: 0.0443 | Smallest expected count: 15.72
The p-value is close to 0.05. Interpret it alongside the estimated lift.
